# Retail Sales Data — Exploratory Data Analysis (EDA)

**Objective:** Explore retail sales patterns, customer behaviour, product performance, and relationships between numerical variables using Python, Pandas, Matplotlib, and Seaborn.

**Dataset:** Kaggle Retail Sales Dataset. The notebook is designed for a transaction-level dataset containing fields such as Date, Customer ID, Gender, Age, Product Category, Quantity, Price per Unit, and Total Amount.

> **Note:** Update `FILE_PATH` below if your downloaded CSV has a different filename.


In [ ]:
# 1. Import libraries and load the dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

FILE_PATH = "Retail Sales Dataset.csv"   # Change this if needed

df = pd.read_csv(FILE_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()


## 1. Initial Inspection

The first step is to understand the size, structure, data types, and completeness of the dataset before performing analysis.


In [ ]:
# Shape, data types, and missing values

print("Shape:", df.shape)

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = df.isnull().sum().to_frame("missing_values")
missing["missing_%"] = (missing["missing_values"] / len(df) * 100).round(2)
display(missing)

print("\nDuplicate rows:", df.duplicated().sum())


### Observation

Review the output above for:
- Dataset size and number of variables.
- Columns stored as numeric, categorical, or date values.
- Columns containing missing values.
- Duplicate transactions that may affect the analysis.

The next step is to standardize column names and convert the date field into a proper datetime format.


In [ ]:
# 2. Data cleaning and preparation

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace(r"[^a-z0-9_]", "", regex=True)
)

# Convert date column if present
date_candidates = ["date", "order_date", "transaction_date", "invoice_date"]
date_col = next((c for c in date_candidates if c in df.columns), None)

if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    print(f"Date column used: {date_col}")
else:
    raise ValueError("No supported date column found. Rename your date column to 'Date'.")

# Convert common numeric fields
for col in ["age", "quantity", "price_per_unit", "total_amount", "sales", "profit", "discount"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Columns:", df.columns.tolist())
df.info()


## 2. Descriptive Statistics

Mean, median, mode, and standard deviation help describe the central tendency and variability of the numerical variables.


In [ ]:
# 3. Descriptive statistics for all numerical columns

numeric_cols = df.select_dtypes(include=np.number).columns

summary = pd.DataFrame({
    "mean": df[numeric_cols].mean(),
    "median": df[numeric_cols].median(),
    "mode": [
        df[col].mode().iloc[0] if not df[col].mode().empty else np.nan
        for col in numeric_cols
    ],
    "std_dev": df[numeric_cols].std()
})

display(summary.round(2))


### Observation

Compare the mean and median for each numerical field. A large difference can indicate skewness or the presence of unusually high or low observations. Standard deviation indicates how widely values vary around their mean.


## 3. Monthly Sales Trend

Monthly aggregation helps identify growth patterns, seasonality, and unusually strong or weak periods.


In [ ]:
# 4. Monthly sales trend

sales_col = "total_amount" if "total_amount" in df.columns else "sales"

monthly_sales = (
    df.set_index(date_col)
      .resample("ME")[sales_col]
      .sum()
)

plt.figure(figsize=(13, 5))
plt.plot(monthly_sales.index, monthly_sales.values, marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Observation

Look for sustained increases/decreases and recurring peaks or dips. These patterns can support inventory planning, promotional timing, and sales-target setting.


## 4. Quarterly Sales Trend

Quarterly aggregation gives management a higher-level view of sales performance.


In [ ]:
# 5. Quarterly sales trend

quarterly_sales = (
    df.set_index(date_col)
      .resample("QE")[sales_col]
      .sum()
)

plt.figure(figsize=(12, 5))
plt.plot(quarterly_sales.index, quarterly_sales.values, marker="o")
plt.title("Quarterly Sales Trend")
plt.xlabel("Quarter")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Observation

Quarterly trends reduce day-to-day noise and make broader business cycles easier to identify. Compare strong and weak quarters and investigate whether the pattern is consistent across years.


## 5. Customer Demographics — Age Groups

Age bands make customer age easier to interpret from a business-segmentation perspective.


In [ ]:
# 6. Age-group distribution

if "age" in df.columns:
    bins = [0, 17, 25, 35, 45, 55, 65, np.inf]
    labels = ["<18", "18–25", "26–35", "36–45", "46–55", "56–65", "66+"]

    df["age_group"] = pd.cut(
        df["age"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    age_counts = df["age_group"].value_counts().sort_index()

    plt.figure(figsize=(10, 5))
    sns.barplot(x=age_counts.index, y=age_counts.values)
    plt.title("Customer Age Group Distribution")
    plt.xlabel("Age Group")
    plt.ylabel("Number of Transactions")
    plt.tight_layout()
    plt.show()
else:
    print("Age column not available.")


### Observation

Identify the age groups contributing the largest number of transactions. This can guide customer targeting, campaign design, and product positioning.


## 6. Gender Breakdown

Gender distribution provides another basic view of the customer base.


In [ ]:
# 7. Gender breakdown

if "gender" in df.columns:
    gender_counts = df["gender"].value_counts()

    plt.figure(figsize=(7, 5))
    sns.barplot(x=gender_counts.index, y=gender_counts.values)
    plt.title("Customer Gender Breakdown")
    plt.xlabel("Gender")
    plt.ylabel("Number of Transactions")
    plt.tight_layout()
    plt.show()

    display(
        (gender_counts / gender_counts.sum() * 100)
        .round(2)
        .rename("percentage")
        .to_frame()
    )
else:
    print("Gender column not available.")


### Observation

Describe whether the transaction base is relatively balanced or concentrated in one gender category. Avoid assuming that transaction share automatically represents the size of the overall customer population.


## 7. Top 10 Best-Selling Products

Product-level analysis identifies items generating the largest sales value.


In [ ]:
# 8. Top 10 products by sales value

product_col = next(
    (c for c in ["product_name", "product", "product_category"] if c in df.columns),
    None
)

if product_col:
    top_products = (
        df.groupby(product_col)[sales_col]
          .sum()
          .sort_values(ascending=False)
          .head(10)
          .sort_values()
    )

    plt.figure(figsize=(11, 6))
    plt.barh(top_products.index.astype(str), top_products.values)
    plt.title("Top 10 Products by Sales")
    plt.xlabel("Sales")
    plt.ylabel("Product")
    plt.tight_layout()
    plt.show()

    display(top_products.sort_values(ascending=False).to_frame("sales"))
else:
    print("Product column not available.")


### Observation

The top products represent the highest sales contributors. These products may deserve stronger availability controls, replenishment planning, and cross-selling opportunities.


## 8. Revenue by Product Category

Category-level analysis shows where overall revenue is concentrated.


In [ ]:
# 9. Revenue by product category

category_col = next(
    (c for c in ["product_category", "category"] if c in df.columns),
    None
)

if category_col:
    category_sales = (
        df.groupby(category_col)[sales_col]
          .sum()
          .sort_values(ascending=False)
    )

    plt.figure(figsize=(9, 5))
    sns.barplot(x=category_sales.index, y=category_sales.values)
    plt.title("Revenue by Product Category")
    plt.xlabel("Product Category")
    plt.ylabel("Revenue")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

    display(category_sales.to_frame("revenue"))
else:
    print("Category column not available.")


### Observation

Identify the category contributing the largest share of revenue and compare it with lower-performing categories. Revenue concentration can influence inventory allocation and category-specific promotions.


## 9. Correlation Heatmap

Correlation helps identify linear relationships among numerical variables. Correlation should not be interpreted as proof of causation.


In [ ]:
# 10. Correlation matrix

corr_cols = df.select_dtypes(include=np.number).columns
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()


### Observation

Focus on strong positive or negative correlations. For example, a strong relationship between quantity and total sales would be expected in many transaction datasets, while weaker relationships can reveal variables that do not independently explain sales very well.


## 10. Additional Visualization — Average Spend by Age Group and Gender

This analysis goes beyond simple demographic counts. It checks whether different customer groups have different average transaction values.


In [ ]:
# 11. Non-obvious insight: average transaction value by age group and gender

if {"age_group", "gender", sales_col}.issubset(df.columns):
    avg_spend = (
        df.groupby(["age_group", "gender"], observed=False)[sales_col]
          .mean()
          .reset_index()
    )

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=avg_spend,
        x="age_group",
        y=sales_col,
        hue="gender"
    )
    plt.title("Average Transaction Value by Age Group and Gender")
    plt.xlabel("Age Group")
    plt.ylabel("Average Transaction Value")
    plt.tight_layout()
    plt.show()

    display(avg_spend.sort_values(sales_col, ascending=False).head(10))
else:
    print("Required demographic columns are not available.")


### Observation

This chart can reveal segments that generate fewer transactions but have higher average transaction values. Such segments may be useful for targeted offers, premium products, or personalized marketing rather than simply maximizing transaction volume.


# 11. Key Business Metrics

A compact KPI view makes the analysis easier to communicate to stakeholders.


In [ ]:
# 12. KPI summary

total_sales = df[sales_col].sum()
total_transactions = len(df)

avg_transaction_value = df[sales_col].mean()

if "quantity" in df.columns:
    total_units = df["quantity"].sum()
else:
    total_units = np.nan

kpis = pd.Series({
    "Total Sales": total_sales,
    "Transactions": total_transactions,
    "Average Transaction Value": avg_transaction_value,
    "Total Units Sold": total_units
})

display(kpis.to_frame("value").round(2))


# 12. Conclusion & Actionable Business Recommendations

Based on the patterns identified in the analysis, recommendations should be tied to the actual results observed in the charts rather than generic assumptions.

### Recommendation 1 — Focus inventory on high-performing products
Maintain adequate stock levels for the top-selling products and monitor their sales trend monthly. Stock-outs in high-demand products can directly affect revenue.

### Recommendation 2 — Use demographic segmentation for targeted campaigns
Use the strongest age/gender transaction segments as a starting point for targeted promotions, while testing whether these groups also have higher average transaction values.

### Recommendation 3 — Plan promotions around seasonal sales patterns
Use the monthly and quarterly trends to identify recurring high-demand periods. Increase inventory and promotional activity before historically strong periods and investigate weak periods for potential corrective action.

### Recommendation 4 — Review category-level performance
Allocate merchandising and marketing resources according to category revenue contribution, while investigating lower-performing categories for pricing, assortment, or promotional improvements.

### Recommendation 5 — Investigate unusual relationships and outliers
Use the correlation matrix and additional segmentation analysis to identify unexpected patterns. High-value or unusual transactions should be investigated before making major business decisions.


## Final Takeaway

This EDA demonstrates a complete analyst workflow:

**Load → Inspect → Clean → Summarize → Analyze trends → Segment customers → Analyze products → Study relationships → Generate business recommendations**

The strongest portfolio projects connect every visualization to a business question and explain *why the finding matters*.
